# Delta Rollback Orchestrator (Bulk Delta Time Travel Operation)

**Objective:** 
This notebook serves as a fail-safe "emergency button" for bulk table recovery. It allows you to simultaneously revert tables across multiple schemas to their last stable version from the previous day (using the `default` input) or any specific day. The goal is to ensure a rapid, auditable recovery in the event that a faulty processing routine or bad deployment introduces corrupted data into the environment.

### 🛠️ How it works
Rather than relying on potentially ambiguous timestamps, the script actively queries the Delta Lake history (`DESCRIBE HISTORY`) for each table. It identifies the exact final version generated on the target date (or yesterday, if using the `default` value) and executes the restore operation based on that precise version ID.

The process is highly **resilient**: if a table does not exist in a given schema, is not in Delta format, or if its older files have already been vacuumed, the script simply logs the failure for that specific target and proceeds with the rollback for the remaining tables.

### 🎛️ Parameters (Widgets)
* **`catalog`**: The target catalog for the operation.
* **`schema_names`**: 
  * `default`: Scans all schemas within the catalog (ignoring system/development schemas defined in the internal blocklist).
  * Comma-separated list (e.g., `schema_name1`, `schema_name2`): Restricts the execution strictly to the provided schemas.
* **`table_names`**: The target table(s) to be restored, provided as a comma-separated list (e.g., `table_name1`, `table_name2`).
* **`execution_mode`**: 
  * `dry_run`: **(Recommended)** Simulation mode. Validates all inputs and rules, and fetches version histories, but DOES NOT execute the final SQL command.
  * `execute`: Performs the actual rollback on the data.
* **`restore_date`**: Defines which day's commit will be used as the restoration anchor point.
  * `default`: Reverts to the last commit of the previous day (default behavior).
  * `YYYY-MM-DD`: Reverts to the last commit of the specific date provided (e.g., `2026-05-10`).
  * *Note: The script automatically rejects future dates and invalid formats.*

### ⚠️ Best Practices
1. Always run in `dry_run` mode first. Review the final summary to confirm that the number of eligible tables aligns with your expectations.
2. The script outputs a detailed summary upon completion, categorizing successes, critical failures, and the specific reasons any tables were ignored. If an error occurs within specific schemas, technical details will be listed in the console output for easy troubleshooting.

In [0]:
from datetime import datetime, timedelta, date
from pyspark.errors import AnalysisException

# Create widgets if necessary (Widgets are Databricks utilities for interactive input in notebooks)
dbutils.widgets.text("catalog", "null", "1. Target Catalog")
dbutils.widgets.text("schema_names", "default", "2. Schemas ('default' or comma-separated)")
dbutils.widgets.text("table_names", "null", "3. Tables (comma-separated)")
dbutils.widgets.dropdown("execution_mode", "dry_run", ["dry_run", "execute"], "4. Execution Mode")
dbutils.widgets.text("restore_date", "default", "5. Target Date ('default' = yesterday, or YYYY-MM-DD)")

# Assign widget values to variables
catalog_name = dbutils.widgets.get("catalog").strip()
raw_schemas = dbutils.widgets.get("schema_names").strip().lower()
execution_mode = dbutils.widgets.get("execution_mode").strip()
raw_table_names = dbutils.widgets.get("table_names").strip()
raw_restore_date = dbutils.widgets.get("restore_date").strip().lower()

# Forbidden characters in inputs
FORBIDDEN_CHARS_STR = "'\"[]{};."
FORBIDDEN_CHARS = set(FORBIDDEN_CHARS_STR)
ERROR_HINT = f"Ensure you do not use the characters: {FORBIDDEN_CHARS_STR} inside the widgets."

In [0]:
def is_delta_table(full_table_name: str) -> bool:
    """
    Checks whether the table uses the Delta format.
    This function intentionally does not use try/except: permission or
    infrastructure errors should propagate to the main loop so they can be
    counted as failures.
    """
    df_detail = spark.sql(f"DESCRIBE DETAIL {full_table_name}")
    table_format = df_detail.select("format").first()[0]
    return table_format.lower() == "delta"


def get_last_version_of_date(full_table_name: str, target_date: date):
    """
    Queries the table's DESCRIBE HISTORY and returns the ID of the latest version
    created strictly on the target date (target_date).
    Returns None only when there are no commits on that date.
    Exceptions (for example, permission issues) are propagated to the main loop.
    """
    start_time = f"{target_date} 00:00:00"
    end_time = f"{target_date} 23:59:59"
    
    query = f"""
        SELECT version
        FROM (DESCRIBE HISTORY {full_table_name})
        WHERE timestamp BETWEEN '{start_time}' AND '{end_time}'
        ORDER BY version DESC
        LIMIT 1
    """
    row = spark.sql(query).first()
    return row.version if row else None


def validate_widget_input(value, field_name: str, empty_hint: str) -> None:
    """
    Validates that the value is not empty and does not contain blocked characters.
    Accepts either a single string or a list of strings.
    """
    values = [value] if isinstance(value, str) else list(value)
    
    if not values or not all(values):
        raise ValueError(f"Widget '{field_name}' is empty or invalid. {empty_hint}")
    
    invalid = [v for v in values if FORBIDDEN_CHARS & set(v)]
    if invalid:
        raise ValueError(
            f"Invalid character(s) in widget '{field_name}': {invalid}\n{ERROR_HINT}"
        )


def resolve_restore_date(value: str) -> date:
    """
    Converts the value from the 'restore_date' widget into a date object.

    - 'default'      -> yesterday (default behavior)
    - 'YYYY-MM-DD'   -> the specific date provided

    Rejects invalid input, including wrong formats, non-existent dates
    (for example, 2026-02-30), and future dates.
    """
    if value == "default":
        return date.today() - timedelta(days=1)
    
    try:
        parsed = datetime.strptime(value, "%Y-%m-%d").date()
    except ValueError:
        raise ValueError(
            f"Invalid value for 'restore_date': '{value}'. "
            f"Use 'default' or a date in the format YYYY-MM-DD (e.g., 2026-05-10)."
        )
    
    if parsed > date.today():
        raise ValueError(
            f"Target date '{value}' is in the future. "
            f"There are no commits to restore for future dates."
        )
    
    return parsed

In [0]:
# Retrieves the widget value, removes leading/trailing spaces, and converts the items into a list using the "," separator
target_tables = [t.strip().lower() for t in raw_table_names.split(",") if t.strip()]
requested_schemas = [c.strip() for c in raw_schemas.split(",") if c.strip()]

# Validates that the widgets are populated and do not contain invalid characters
validate_widget_input(catalog_name, "catalog", "Please specify the target catalog.")
validate_widget_input(target_tables, "table_names", "Please specify at least one table.")
validate_widget_input(requested_schemas, "schema_names", "Enter 'default' or a comma-separated list of schemas.")
validate_widget_input(raw_restore_date, "restore_date", "Enter 'default' or a date in YYYY-MM-DD format.")

# Resolves the target date for the RESTORE (default -> yesterday, or a specific date)
restore_date = resolve_restore_date(raw_restore_date)

In [0]:
schema_blocklist = {
    "bronze", "silver", "gold", "mockup_scale", "default", 
    "information_schema", "sys", "system", "temp"
}

print(f"--- ROLLBACK PREPARATION ---")
print(f"Catalog: {catalog_name} | Mode: {execution_mode.upper()}")
print(f"Target Tables ({len(target_tables)}): {', '.join(target_tables)}")

# Clearly indicates which date will be used (and whether it came from the default or manual input)
date_origin = "default = yesterday" if raw_restore_date == "default" else "manual"
print(f"\nTarget RESTORE Date: {restore_date} ({date_origin})")

try:
    df_schemas_found = spark.sql(f"SHOW SCHEMAS IN {catalog_name}")
    col_name = df_schemas_found.columns[0] 
    
    all_schemas = [str(row[col_name]).lower() for row in df_schemas_found.collect()]
    valid_schemas = [s for s in all_schemas if s not in schema_blocklist]
    
    if raw_schemas == "default":
        target_schemas = valid_schemas
        print(f"-> BATCH Execution (Default). Found {len(target_schemas)} eligible schemas.")
        
    else:          
        target_schemas = []
        invalid_schemas = []
        
        for schema in requested_schemas:
            if schema in valid_schemas:
                target_schemas.append(schema)
            else:
                invalid_schemas.append(schema)
                
        if invalid_schemas:
            raise ValueError(f"ERROR: The following schemas do not exist in the catalog or are on the blocklist: {', '.join(invalid_schemas)}")
            
        print(f"-> RESTRICTED Execution to {len(target_schemas)} schema(s): {', '.join(target_schemas)}")

except Exception as e:
    print(f"Critical error during schema validation.")
    raise e

In [0]:
print("\n--- STARTING PROCESSING ---")
success_count = 0
failure_count = 0
skipped_not_found_count = 0
skipped_not_delta_count = 0
skipped_no_commit_count = 0
failed_tables = []

for schema in target_schemas:
    print(f"\n[SCHEMA: {schema.upper()}] ==============================")
    
    for table_name in target_tables:
        full_table_name = f"{catalog_name}.{schema}.{table_name}"
        
        try:
            if not spark.catalog.tableExists(full_table_name):
                print(f"  [IGNORED] {full_table_name}: Does not exist in this schema.")
                skipped_not_found_count += 1
                continue
                
            if not is_delta_table(full_table_name):
                print(f"  [IGNORED] {full_table_name}: Exists, but is not in Delta format.")
                skipped_not_delta_count += 1
                continue
                
            target_version = get_last_version_of_date(full_table_name, restore_date)
            
            if target_version is None:
                print(f"  [IGNORED] {full_table_name}: No commit found on {restore_date}.")
                skipped_no_commit_count += 1
                continue
                
            restore_query = f"RESTORE TABLE {full_table_name} TO VERSION AS OF {target_version}"
            
            if execution_mode == "execute":
                spark.sql(restore_query)
                print(f"  [SUCCESS]  {full_table_name}: Restored to version {target_version}")
                success_count += 1
            else:
                print(f"  [DRY RUN]  {full_table_name}: Command validated for version {target_version}")
                success_count += 1
                
        except AnalysisException as ae:
            print(f"  [DELTA ERROR] {full_table_name}: Internal Delta failure.")
            failure_count += 1
            failed_tables.append({"table_name": full_table_name, "error_message": str(ae)})
        except Exception as e:
            print(f"  [GENERAL ERROR] {full_table_name}: Unexpected failure.")
            failure_count += 1
            failed_tables.append({"table_name": full_table_name, "error_message": str(e)})

In [0]:
total_operations = success_count + failure_count + skipped_not_found_count + skipped_not_delta_count + skipped_no_commit_count

print("\n=======================================================")
print(f"       SUMMARY ({execution_mode.upper()})        ")
print("=======================================================")
print(f"Total operations analyzed: {total_operations}")

if execution_mode == "dry_run":
    print(f"  ✓ Validated tables that WOULD BE restored: {success_count}")
else:
    print(f"  ✓ Successfully restored tables: {success_count}")

print(f"  ✗ Critical Failures: {failure_count}")
print(f"  ○ Ignored (not found): {skipped_not_found_count}")
print(f"  ○ Ignored (not Delta): {skipped_not_delta_count}")
print(f"  ○ Ignored (no commit on {restore_date}): {skipped_no_commit_count}")

if failure_count > 0:
    print("\n⚠️ THE FOLLOWING TABLES FAILED:")
    print("-" * 60)
    for failure in failed_tables:
        print(f"Target: {failure['table_name']}")
        print(f"Error: {failure['error_message']}\n")
    print("-" * 60)